# MEDAL encoder interpretability

In [1]:
import medal
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import pickle
import shap
import torch.nn as nn
import torch

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

from functools import partial

/Users/clairehe/anaconda3/envs/medal_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-09-24 18:13:47,629	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-09-24 18:13:48,225	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## Exploration on simulated data

## Exploration on baby MNIST

In [2]:
from sklearn.datasets import load_digits 

data = load_digits()
X, y = data['data'], data['target']
X_train, X_test, y_trian, y_test = train_test_split(X, y, test_size=0.2)

In [ ]:
# 1. Find the best autoencoder architecture for your data
result = medal.tune_medal_architecture(
    X_train,
    teacher="tsne",
    teacher_params={"perplexity": 30},
    output_dir="experiments/arch_search",
)
arch_config = {
    "hidden_dims": [512, 512],
    "latent_dim": 2,
    "activation": "SELU",
    "use_batchnorm": False,
    "dropout_rate": 0.0,
    "lr": 0.01,
    "lambda_d": 5000,
    "max_epochs": 3000,
    "batch_size": 256,
}

# 2. Sweep teacher hyperparameters, training one student per setting
sweep_results = medal.run_teacher_sweep(
    X_train,
    output_dir="experiments/sweep",
    teacher="tsne",
    arch_config=arch_config,
)

# 3. Evaluate and select the optimal hyperparameter
df = sweep_results.load_metrics(X_test)
opt_param = medal.select_teacher_param(df, param_col="perplexity")
print(f"Optimal perplexity: {opt_param}")

# 4. Visualise reconstruction error across the sweep
medal.plot_reconstruction_error(df, opt_param, param_col="perplexity")

(_arch_search_trainable pid=7252) Exception ignored in: <function WeakSet.__init__.<locals>._remove at 0x1015bcd30>
(_arch_search_trainable pid=7252) Traceback (most recent call last):
(_arch_search_trainable pid=7252)   File "/Users/clairehe/anaconda3/envs/medal_env/lib/python3.10/_weakrefset.py", line 39, in _remove
(_arch_search_trainable pid=7252)     def _remove(item, selfref=ref(self)):
(_arch_search_trainable pid=7252)   File "/Users/clairehe/anaconda3/envs/medal_env/lib/python3.10/site-packages/ray/_private/worker.py", line 1060, in sigterm_handler
(_arch_search_trainable pid=7252)     raise_sys_exit_with_custom_error_message(
(_arch_search_trainable pid=7252)   File "python/ray/_raylet.pyx", line 714, in ray._raylet.raise_sys_exit_with_custom_error_message
(_arch_search_trainable pid=7252) SystemExit: 1
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or Inf found in input tensor.
NaN or